# Contest MVP Dataset Selection

## tl;dr

- Keep **four required launch datasets**: SECOP II contracts, SECOP II procurement processes, SECOP II contract suspensions, and PACO sanctions/red flags.
- Allow **one conditional fifth source**: SECOP contract modifications, only after a full historical backfill and model ablation pass. It would add three variables, taking the model from 15 to 18.
- Do not promote RUES, TVEC, BPIN spending, or other sparse enrichments into the core feature matrix. Their measured coverage is too low or their incremental model value is unproven.
- The process source joins to **5,437,369 of 5,441,243** contract records (99.93%) through `id_del_portafolio`; on contracts signed since 2023 the match is **3,265,784 of 3,269,620** (99.88%).


## Context & Methods

This notebook supports a contest-scope decision: add only sources that materially improve a present-day contract risk-prioritization MVP without creating a new infrastructure project.

### Key Assumptions

- The product prioritizes contracts **as of the scoring snapshot**; it does not claim to predict corruption at award time.
- Exact contract/process/company identifiers are required. Fuzzy person matching is out of scope.
- Raw lake sources contain repeated snapshots. Event features must deduplicate source records before aggregation.
- The bounded launch cohort is contracts signed on or after **2023-01-01**.


In [1]:
from pathlib import Path
import json
import duckdb
import pandas as pd
from IPython.display import display

REPO = Path.cwd()
REALITY_PATH = REPO / "lake/meta/reality/2026-06-05.json"
AWARDS_GLOB = "lake/curated/table=fct_procurement_contract_awards/*.parquet"
PROCESS_GLOB = "lake/raw/source=p6dx-8zbt/**/*.parquet"
SUSPENSION_GLOB = "lake/raw/source=u99c-7mfm/**/*.parquet"
MODIFICATION_GLOB = "lake/raw/source=u8cx-r425/**/*.parquet"
EXECUTION_GLOB = "lake/raw/source=mfmm-jqmq/**/*.parquet"

con = duckdb.connect()
con.execute("PRAGMA threads=4")
reality = json.loads(REALITY_PATH.read_text(encoding="utf-8"))


## Data

In [2]:
candidate_ids = [
    "jbjy-vk9h", "p6dx-8zbt", "wi7w-2nvm", "u8cx-r425",
    "u99c-7mfm", "mfmm-jqmq", "it5q-hg94", "paco_sanctions",
    "c82u-588k", "gbry-rnq4", "jgra-rz2t",
]
source_roles = {
    "jbjy-vk9h": "core contract spine",
    "p6dx-8zbt": "competition and price context",
    "wi7w-2nvm": "offer-level competition",
    "u8cx-r425": "post-award modifications",
    "u99c-7mfm": "post-award suspensions",
    "mfmm-jqmq": "execution progress",
    "it5q-hg94": "sanction outcomes",
    "paco_sanctions": "weak labels and official evidence",
    "c82u-588k": "company registry context",
    "gbry-rnq4": "person conflict disclosures",
    "jgra-rz2t": "2019 campaign finance",
}
by_id = {row["dataset_id"]: row for row in reality["datasets"]}
rows = []
for dataset_id in candidate_ids:
    item = by_id[dataset_id]
    raw_root = REPO / f"lake/raw/source={dataset_id}"
    raw_bytes = sum(p.stat().st_size for p in raw_root.rglob("*.parquet"))
    rows.append({
        "dataset_id": dataset_id,
        "dataset": item["dataset_name"],
        "role": source_roles[dataset_id],
        "raw_rows": item["row_count"],
        "raw_size_gib": raw_bytes / (1024 ** 3),
        "watermark_min": item["watermark_min"],
        "watermark_max": item["watermark_max"],
        "raw_duplicate_rate": item["dup_ratio"],
        "join_key": ", ".join(item["join_key_columns"]),
    })
candidate_inventory = pd.DataFrame(rows)
display(candidate_inventory.round({"raw_size_gib": 3, "raw_duplicate_rate": 4}))


,dataset_id,dataset,role,raw_rows,raw_size_gib,watermark_min,watermark_max,raw_duplicate_rate,join_key
0,jbjy-vk9h,SECOP II - Contratos Electrónicos,core contract spine,5615290,1.222,2015-06-11T00:00:00.000,2026-05-04T00:00:00.000,0.0001,contract_id
1,p6dx-8zbt,SECOP II - Procesos de Contratación,competition and price context,8648159,1.273,2015-04-16T00:00:00.000,2026-05-21T00:00:00.000,0.9395,id_adjudicacion
2,wi7w-2nvm,SECOPII - Ofertas Por Proceso,offer-level competition,42264359,0.975,2022-12-31T00:00:00.000,2026-04-21T00:00:00.000,0.9940,id_del_proceso_de_compra
3,u8cx-r425,SECOP II - Modificaciones a contratos,post-award modifications,63000,0.010,2026-06-03T00:00:00.000,2026-06-03T00:00:00.000,0.8983,id_contrato
4,u99c-7mfm,SECOP II - Suspensiones de Contratos,post-award suspensions,537314,0.013,2016-05-11T00:00:00.000,2026-06-03T00:00:00.000,0.8446,id_contrato
5,mfmm-jqmq,SECOP II - Ejecución Contratos,execution progress,9535,0.000,2026-05-29T00:00:00.000,2026-06-04T00:00:00.000,0.2451,identificadorcontrato
6,it5q-hg94,SECOPII - Multas y Sanciones,sanction outcomes,538,0.001,2018-07-04T00:00:00.000,2028-07-22T00:00:00.000,0.2546,contract_id
7,paco_sanctions,PACO - sanciones y red flags,weak labels and official evidence,54369,0.004,NaN,NaN,0.6144,subject_document_id
8,c82u-588k,Personas Naturales Personas Jurídicas y Entida...,company registry context,9269440,0.596,1900/01/01 00:00:00.000000000,2026/05/04 12:17:08.880000000,0.1426,document_id
9,gbry-rnq4,Declaraciones conflictos de interés,person conflict disclosures,328799,0.011,2022-01-01T00:03:57.182,2022-12-13T14:57:31.146,0.2739,document_id


Raw duplicate rates describe repeated lake snapshots, not necessarily duplicate business events. Candidate event sources are evaluated after deterministic deduplication at their intended grain.

## Results

In [3]:
award_base_sql = f"""
SELECT count(*) AS row_count,
       count(DISTINCT contract_id) AS contracts,
       count(DISTINCT process_id) AS processes
FROM read_parquet('{AWARDS_GLOB}')
"""
process_join_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id, nullif(trim(process_id), '') AS process_id
  FROM read_parquet('{AWARDS_GLOB}')
), processes AS (
  SELECT DISTINCT nullif(trim(id_del_portafolio), '') AS process_id
  FROM read_parquet('{PROCESS_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE nullif(trim(id_del_portafolio), '') IS NOT NULL
)
SELECT count(*) AS award_contracts,
       count(*) FILTER (WHERE processes.process_id IS NOT NULL) AS matched_contracts
FROM awards LEFT JOIN processes USING (process_id)
"""
modification_join_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id FROM read_parquet('{AWARDS_GLOB}')
), events AS (
  SELECT nullif(trim(id_contrato), '') AS contract_id, count(*) AS event_count
  FROM (
    SELECT DISTINCT id_contrato, identificador_modificacion, identificador,
           identificador_requerimiento, valor_modificacion, fecha_de_aprobacion
    FROM read_parquet('{MODIFICATION_GLOB}', hive_partitioning=false)
  )
  WHERE nullif(trim(id_contrato), '') IS NOT NULL
  GROUP BY 1
)
SELECT count(*) AS source_contracts,
       sum(event_count) AS deduplicated_events,
       count(*) FILTER (WHERE awards.contract_id IS NOT NULL) AS matched_contracts
FROM events LEFT JOIN awards USING (contract_id)
"""
suspension_join_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id FROM read_parquet('{AWARDS_GLOB}')
), events AS (
  SELECT nullif(trim(id_contrato), '') AS contract_id, count(*) AS event_count
  FROM (
    SELECT DISTINCT id_contrato, tipo, fecha_de_creacion, fecha_de_aprobacion,
           proposito_de_la_modificacion, fecha_de_inicio_del_contrato,
           fecha_de_fin_del_contrato
    FROM read_parquet('{SUSPENSION_GLOB}', hive_partitioning=false)
  )
  WHERE nullif(trim(id_contrato), '') IS NOT NULL
  GROUP BY 1
)
SELECT count(*) AS source_contracts,
       sum(event_count) AS deduplicated_events,
       count(*) FILTER (WHERE awards.contract_id IS NOT NULL) AS matched_contracts
FROM events LEFT JOIN awards USING (contract_id)
"""
execution_join_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id FROM read_parquet('{AWARDS_GLOB}')
), items AS (
  SELECT nullif(trim(identificadorcontrato), '') AS contract_id, count(*) AS item_count
  FROM (
    SELECT DISTINCT identificadorcontrato, tipoejecucion, nombreplan,
           fechadeentregaesperada, porcentajedeavanceesperado, fechadeentregareal,
           porcentaje_de_avance_real, estado_del_contrato, referencia_de_articulos,
           descripci_n, unidad, cantidad_adjudicada, cantidad_planeada,
           cantidadrecibida, cantidadporrecibir, fechacreacion
    FROM read_parquet('{EXECUTION_GLOB}', hive_partitioning=false)
  )
  WHERE nullif(trim(identificadorcontrato), '') IS NOT NULL
  GROUP BY 1
)
SELECT count(*) AS source_contracts,
       sum(item_count) AS deduplicated_items,
       count(*) FILTER (WHERE awards.contract_id IS NOT NULL) AS matched_contracts
FROM items LEFT JOIN awards USING (contract_id)
"""

checks = {
    "award_base": con.execute(award_base_sql).fetchdf().iloc[0].to_dict(),
    "process": con.execute(process_join_sql).fetchdf().iloc[0].to_dict(),
    "modifications": con.execute(modification_join_sql).fetchdf().iloc[0].to_dict(),
    "suspensions": con.execute(suspension_join_sql).fetchdf().iloc[0].to_dict(),
    "execution": con.execute(execution_join_sql).fetchdf().iloc[0].to_dict(),
}
join_rows = []
for source in ["process", "modifications", "suspensions", "execution"]:
    values = checks[source]
    denominator = values.get("award_contracts", values.get("source_contracts"))
    join_rows.append({
        "source": source,
        "denominator_contracts": int(denominator),
        "matched_contracts": int(values["matched_contracts"]),
        "key_match_rate": values["matched_contracts"] / denominator,
    })
join_quality = pd.DataFrame(join_rows)
display(join_quality.assign(key_match_rate=join_quality["key_match_rate"].map("{:.2%}".format)))


,source,denominator_contracts,matched_contracts,key_match_rate
0,process,5441243,5437369,99.93%
1,modifications,6407,6109,95.35%
2,suspensions,83494,79168,94.82%
3,execution,7198,7069,98.21%


In [4]:
process_completeness_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id, nullif(trim(process_id), '') AS process_id
  FROM read_parquet('{AWARDS_GLOB}')
  WHERE signing_date >= DATE '2023-01-01'
), processes AS (
  SELECT nullif(trim(id_del_portafolio), '') AS process_id,
         bool_or(coalesce(nullif(trim(conteo_de_respuestas_a_ofertas), ''),
                          nullif(trim(respuestas_al_procedimiento), ''),
                          nullif(trim(respuestas_externas), ''),
                          nullif(trim(proveedores_unicos_con), '')) IS NOT NULL) AS has_responses,
         bool_or(coalesce(nullif(trim(proveedores_invitados), ''),
                          nullif(trim(proveedores_con_invitacion), '')) IS NOT NULL) AS has_invited,
         bool_or(coalesce(nullif(trim(fecha_de_apertura_efectiva), ''),
                          nullif(trim(fecha_de_apertura_de_respuesta), '')) IS NOT NULL
                 AND nullif(trim(fecha_de_recepcion_de), '') IS NOT NULL) AS has_window,
         bool_or(nullif(trim(precio_base), '') IS NOT NULL
                 AND nullif(trim(valor_total_adjudicacion), '') IS NOT NULL) AS has_price_pair
  FROM read_parquet('{PROCESS_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE nullif(trim(id_del_portafolio), '') IS NOT NULL
  GROUP BY 1
)
SELECT count(*) AS contracts,
       count(*) FILTER (WHERE processes.process_id IS NOT NULL) AS process_match,
       count(*) FILTER (WHERE has_responses) AS response_count,
       count(*) FILTER (WHERE has_invited) AS invited_count,
       count(*) FILTER (WHERE has_price_pair) AS price_pair,
       count(*) FILTER (WHERE has_window) AS bidding_window
FROM awards LEFT JOIN processes USING (process_id)
"""
recent = con.execute(process_completeness_sql).fetchdf().iloc[0]
feature_completeness = pd.DataFrame([
    {"feature_group": label, "available_contracts": int(recent[field]),
     "total_contracts": int(recent["contracts"]),
     "availability_rate": recent[field] / recent["contracts"]}
    for label, field in [
        ("Process record match", "process_match"),
        ("Response count", "response_count"),
        ("Invited supplier count", "invited_count"),
        ("Award/base price pair", "price_pair"),
        ("Bidding window timestamps", "bidding_window"),
    ]
])
display(feature_completeness.assign(availability_rate=feature_completeness["availability_rate"].map("{:.2%}".format)))


,feature_group,available_contracts,total_contracts,availability_rate
0,Process record match,3265784,3269620,99.88%
1,Response count,3265784,3269620,99.88%
2,Invited supplier count,3265784,3269620,99.88%
3,Award/base price pair,3265784,3269620,99.88%
4,Bidding window timestamps,263552,3269620,8.06%


### Extension candidate checks

These checks test whether additional exact-key sources are broad and complete enough to become required model inputs. Coverage uses the most relevant denominator for each source and is not treated as a common performance score.

In [5]:
RUES_GLOB = "lake/raw/source=c82u-588k/**/*.parquet"
TVEC_GLOB = "lake/raw/source=3hdv-smhz/**/*.parquet"
BPIN_LINK_GLOB = "lake/raw/source=d9na-abhe/**/*.parquet"
SGR_EXPENSE_GLOB = "lake/raw/source=qkv4-ek54/**/*.parquet"

rues_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id, supplier_nit_base, signing_date
  FROM read_parquet('{AWARDS_GLOB}')
  WHERE signing_date >= DATE '2023-01-01' AND supplier_nit_base IS NOT NULL
), rues_rows AS (
  SELECT regexp_replace(document_id, '[^0-9]', '', 'g') AS nit_base,
         upper(trim(coalesce(matricula_status, ''))) AS status,
         try_strptime(matricula_date, '%Y%m%d')::DATE AS matricula_date,
         try_cast(last_renewed_year AS INTEGER) AS renewed_year
  FROM read_parquet('{RUES_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE upper(trim(coalesce(identification_class, ''))) = 'NIT'
    AND length(regexp_replace(document_id, '[^0-9]', '', 'g')) = 9
    AND NOT regexp_matches(regexp_replace(document_id, '[^0-9]', '', 'g'), '^0+$')
), rues AS (
  SELECT nit_base,
         bool_or(status = 'ACTIVA') AS any_active,
         min(matricula_date) FILTER (WHERE matricula_date BETWEEN DATE '1900-01-01' AND DATE '2026-07-13') AS first_matricula_date,
         max(renewed_year) FILTER (WHERE renewed_year BETWEEN 1900 AND 2026) AS last_renewed_year
  FROM rues_rows GROUP BY 1
)
SELECT count(*) AS eligible_contracts,
       count(DISTINCT awards.supplier_nit_base) AS eligible_suppliers,
       count(*) FILTER (WHERE rues.nit_base IS NOT NULL) AS matched_contracts,
       count(DISTINCT awards.supplier_nit_base) FILTER (WHERE rues.nit_base IS NOT NULL) AS matched_suppliers
FROM awards LEFT JOIN rues ON rues.nit_base = awards.supplier_nit_base
"""

tvec_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id, supplier_nit_base
  FROM read_parquet('{AWARDS_GLOB}')
  WHERE signing_date >= DATE '2023-01-01' AND supplier_nit_base IS NOT NULL
), suppliers AS (
  SELECT regexp_replace(coalesce(supplier_nit, ''), '[^0-9]', '', 'g') AS supplier_nit_base
  FROM read_parquet('{TVEC_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE length(regexp_replace(coalesce(supplier_nit, ''), '[^0-9]', '', 'g')) = 9
  GROUP BY 1
)
SELECT count(*) AS eligible_contracts,
       count(DISTINCT awards.supplier_nit_base) AS eligible_suppliers,
       count(*) FILTER (WHERE suppliers.supplier_nit_base IS NOT NULL) AS matched_contracts,
       count(DISTINCT awards.supplier_nit_base) FILTER (WHERE suppliers.supplier_nit_base IS NOT NULL) AS matched_suppliers
FROM awards LEFT JOIN suppliers USING (supplier_nit_base)
"""

bpin_sql = f"""
WITH awards AS (
  SELECT DISTINCT contract_id
  FROM read_parquet('{AWARDS_GLOB}')
  WHERE signing_date >= DATE '2023-01-01'
), links AS (
  SELECT DISTINCT nullif(trim(id_contracto), '') AS contract_id,
         regexp_replace(coalesce(codigo_bpin, ''), '[^0-9]', '', 'g') AS bpin
  FROM read_parquet('{BPIN_LINK_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE nullif(trim(id_contracto), '') IS NOT NULL
), expenses AS (
  SELECT DISTINCT regexp_replace(coalesce(bpin, ''), '[^0-9]', '', 'g') AS bpin
  FROM read_parquet('{SGR_EXPENSE_GLOB}', hive_partitioning=false, union_by_name=true)
  WHERE regexp_replace(coalesce(bpin, ''), '[^0-9]', '', 'g') <> ''
)
SELECT count(DISTINCT awards.contract_id) AS cohort_contracts,
       count(DISTINCT awards.contract_id) FILTER (WHERE links.bpin <> '') AS linked_contracts,
       count(DISTINCT awards.contract_id) FILTER (WHERE expenses.bpin IS NOT NULL) AS expense_matched_contracts
FROM awards LEFT JOIN links USING (contract_id) LEFT JOIN expenses USING (bpin)
"""

rues_check = con.execute(rues_sql).fetchdf().iloc[0].to_dict()
tvec_check = con.execute(tvec_sql).fetchdf().iloc[0].to_dict()
bpin_check = con.execute(bpin_sql).fetchdf().iloc[0].to_dict()
modification_check = join_quality.loc[join_quality['source'] == 'modifications'].iloc[0].to_dict()

extension_checks = pd.DataFrame([
    {"candidate": "Contract modifications", "coverage_rate": modification_check["key_match_rate"], "denominator": "loaded modification-event contracts", "decision": "Conditional fifth", "blocker": "Local history covers only 2026-06-03"},
    {"candidate": "RUES company registry", "coverage_rate": rues_check["matched_contracts"] / rues_check["eligible_contracts"], "denominator": "2023+ exact-NIT contract rows", "decision": "Enrichment only", "blocker": "60.1% of eligible contract rows unmatched"},
    {"candidate": "TVEC supplier history", "coverage_rate": tvec_check["matched_contracts"] / tvec_check["eligible_contracts"], "denominator": "2023+ exact-NIT contract rows", "decision": "Defer", "blocker": "Under 1% contract-row coverage"},
    {"candidate": "BPIN project link", "coverage_rate": bpin_check["linked_contracts"] / bpin_check["cohort_contracts"], "denominator": "all 2023+ contracts", "decision": "Evidence only", "blocker": "Project link is broad, but incremental risk value is unproven"},
    {"candidate": "SGR expense execution", "coverage_rate": bpin_check["expense_matched_contracts"] / bpin_check["cohort_contracts"], "denominator": "all 2023+ contracts", "decision": "Defer", "blocker": "Expense match reaches only 0.39%"},
])
display(extension_checks.assign(coverage_rate=extension_checks["coverage_rate"].map("{:.2%}".format)))


,candidate,coverage_rate,denominator,decision,blocker
0,Contract modifications,95.35%,loaded modification-event contracts,Conditional fifth,Local history covers only 2026-06-03
1,RUES company registry,39.89%,2023+ exact-NIT contract rows,Enrichment only,60.1% of eligible contract rows unmatched
2,TVEC supplier history,0.95%,2023+ exact-NIT contract rows,Defer,Under 1% contract-row coverage
3,BPIN project link,52.65%,all 2023+ contracts,Evidence only,"Project link is broad, but incremental risk va..."
4,SGR expense execution,0.39%,all 2023+ contracts,Defer,Expense match reaches only 0.39%


In [6]:
decision = pd.DataFrame([
    {"priority": 1, "dataset_id": "jbjy-vk9h", "decision": "Launch", "reason": "Contract spine and nine established behavior variables"},
    {"priority": 2, "dataset_id": "p6dx-8zbt", "decision": "Launch", "reason": "99.93% contract join; four near-complete competition/price variables"},
    {"priority": 3, "dataset_id": "u99c-7mfm", "decision": "Launch", "reason": "Full 2016-2026 window; two interpretable disruption variables"},
    {"priority": 4, "dataset_id": "paco_sanctions", "decision": "Launch", "reason": "Exact-NIT evidence and weak evaluation labels; never a model input"},
    {"priority": 5, "dataset_id": "u8cx-r425", "decision": "After backfill", "reason": "High-value signal, but local ingest is only a recent 63k-row slice"},
    {"priority": 6, "dataset_id": "mfmm-jqmq", "decision": "After backfill", "reason": "Only 9,535 local rows from a one-week creation window"},
    {"priority": 7, "dataset_id": "wi7w-2nvm", "decision": "Defer", "reason": "42.3m raw rows; process response counts cover the MVP need"},
    {"priority": 8, "dataset_id": "c82u-588k", "decision": "Defer", "reason": "Useful registry context, but adds 9.3m rows and new feature semantics"},
    {"priority": 9, "dataset_id": "it5q-hg94", "decision": "Defer", "reason": "Tiny local slice and a future-dated watermark require quality repair"},
    {"priority": 10, "dataset_id": "gbry-rnq4", "decision": "Exclude", "reason": "Person-level privacy scope and stale 2022 window"},
    {"priority": 11, "dataset_id": "jgra-rz2t", "decision": "Exclude", "reason": "2019-only scope and invalid date outliers"},
])
display(decision)

metrics = {
    "snapshot_date": reality["snapshot_date"],
    "award_base": checks["award_base"],
    "join_quality": join_quality.to_dict(orient="records"),
    "recent_process_feature_completeness": feature_completeness.to_dict(orient="records"),
    "decision": decision.to_dict(orient="records"),
    "extension_checks": extension_checks.to_dict(orient="records"),
}
metrics_path = REPO / "docs/analysis/dataset_selection_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, default=str), encoding="utf-8")
print(f"Wrote {metrics_path.relative_to(REPO)}")


,priority,dataset_id,decision,reason
0,1,jbjy-vk9h,Launch,Contract spine and nine established behavior v...
1,2,p6dx-8zbt,Launch,99.93% contract join; four near-complete compe...
2,3,u99c-7mfm,Launch,Full 2016-2026 window; two interpretable disru...
3,4,paco_sanctions,Launch,Exact-NIT evidence and weak evaluation labels;...
4,5,u8cx-r425,After backfill,"High-value signal, but local ingest is only a ..."
5,6,mfmm-jqmq,After backfill,"Only 9,535 local rows from a one-week creation..."
6,7,wi7w-2nvm,Defer,42.3m raw rows; process response counts cover ...
7,8,c82u-588k,Defer,"Useful registry context, but adds 9.3m rows an..."
8,9,it5q-hg94,Defer,Tiny local slice and a future-dated watermark ...
9,10,gbry-rnq4,Exclude,Person-level privacy scope and stale 2022 window


Wrote docs/analysis/dataset_selection_metrics.json


## Takeaways

1. **Four sources remain the required MVP.** They lift the entry into the contest's intermediate 3–10 source band without introducing graph infrastructure or unbounded feature work.
2. **One fifth source is defensible only behind gates.** Contract modifications have a 95.35% exact-key match on the loaded event slice and already support interpretable rollups, but the local ingest covers only June 3, 2026. A full backfill must prove that absence can safely mean zero.
3. **The other enrichments are too sparse for core model input.** RUES matches 39.89% of exact-NIT contract rows; TVEC matches 0.95%; BPIN links cover 52.65% of all recent contracts, but SGR expense records reach only 0.39%.
4. **Process data should replace offers for the MVP.** Response, invitation, and price fields are available for 99.88% of the bounded 2023+ cohort. Bidding-window timestamps are available for only about 8%, so that field is excluded.
5. **PACO remains outside the feature matrix.** It supports exact-NIT documentary evidence and supplier-disjoint evaluation only, avoiding direct label leakage.
